# Compute Cornerness Score

In [1]:
%pip install numpy

Note: you may need to restart the kernel to use updated packages.


In [2]:
import numpy as np

### Cornerness Score function

In [3]:
# Copy from 1_1_gaussian_filtering.ipynb
# TODO: this should be imported from a shared module instead of copy-pasted
def gaussian_filtering(image, image_width, image_height):
    # fixed Gaussian kernel
    gaussian_kernel = np.array([[1, 2, 1], [2, 4, 2], [1, 2, 1]]) / 16
    kernel_size = gaussian_kernel.shape[0]

    extended_image = np.pad(image, pad_width=1, mode="edge")

    filtered_image = np.zeros_like(image, dtype=np.float64)
    for i in range(image_height):
        for j in range(image_width):
            region = extended_image[i : i + kernel_size, j : j + kernel_size]
            filtered_image[i, j] = np.sum(region * gaussian_kernel)
    return filtered_image


In [16]:
def compute_cornerness_score(ix_square, iy_square, ixiy, alpha, threshold, image_width, image_height):
    top = 1000

    # top 1000 corners, represented as a list of tuples containing the x and y coordinates of each corner and its corresponding cornerness score
    corners = []
    for i in range(image_height):
        for j in range(image_width):
            H_ix = ix_square[i, j]
            H_iy = iy_square[i, j]
            H_ixiy = ixiy[i, j]
            # Compute cornerness score using the formula: R = det(H) - alpha * (trace(H))^2
            det_H = H_ix * H_iy - H_ixiy ** 2
            trace_H = H_ix + H_iy
            score = det_H - alpha * (trace_H ** 2)
            
            if score > threshold:
                corners.append((j, i, score))  # (x, y, cornerness score)
    corners.sort(key=lambda x: x[2], reverse=True)  # Sort by cornerness score in descending order
    return corners[:top]  # Return top 1000 corners

### Load images derivative

In [5]:
image_width = 210
image_height = 200

prefix = "assets/cornerness_score_expected_outputs/step3_blurred_"
left_ix_square = np.load(prefix + "left_ix_square.npy")
left_iy_square = np.load(prefix + "left_iy_square.npy")
left_ixiy = np.load(prefix + "left_ixiy.npy")
right_ix_square = np.load(prefix + "right_ix_square.npy")
right_iy_square = np.load(prefix + "right_iy_square.npy")
right_ixiy = np.load(prefix + "right_ixiy.npy")

#### Verify blurred Ix2, Iy2, IxIy

In [6]:
prefix = "assets/sobel_derivative_expected_outputs/step2_"
compute_left_ix_square = gaussian_filtering(np.load(prefix + "left_ix_square.npy"), image_width, image_height)
compute_left_iy_square = gaussian_filtering(np.load(prefix + "left_iy_square.npy"), image_width, image_height)
compute_left_ixiy = gaussian_filtering(np.load(prefix + "left_ixiy.npy"), image_width, image_height)
compute_right_ix_square = gaussian_filtering(np.load(prefix + "right_ix_square.npy"), image_width, image_height)
compute_right_iy_square = gaussian_filtering(np.load(prefix + "right_iy_square.npy"), image_width, image_height)
compute_right_ixiy = gaussian_filtering(np.load(prefix + "right_ixiy.npy"), image_width, image_height)

print(np.array_equal(left_ix_square, compute_left_ix_square))
print(np.array_equal(left_iy_square, compute_left_iy_square))
print(np.array_equal(left_ixiy, compute_left_ixiy))
print(np.array_equal(right_ix_square, compute_right_ix_square))
print(np.array_equal(right_iy_square, compute_right_iy_square))
print(np.array_equal(right_ixiy, compute_right_ixiy))

True
True
True
True
True
True


### Calculate cornerness scores

In [17]:
alpha = 0.04
threshold = 10 ** 6

cornerness_score_left = compute_cornerness_score(left_ix_square, left_iy_square, left_ixiy, alpha, threshold, image_width, image_height)
cornerness_score_right = compute_cornerness_score(right_ix_square, right_iy_square, right_ixiy, alpha, threshold, image_width, image_height)

### Verfiy the result by comparing sample filtered images

In [18]:
print(len(cornerness_score_left))  # Print the number of corners for the left image
print(len(cornerness_score_right))  # Print the number of corners for the right image
print(cornerness_score_left[:5])  # Print the top 5 corners for the left image
print(cornerness_score_right[:5])  # Print the top 5 corners for the right image

3
3
[(70, 107, np.float64(1054227.8537187194)), (193, 150, np.float64(1025871.1260015869)), (193, 149, np.float64(1000943.1167047119))]
[(170, 47, np.float64(1055012.9793435668)), (123, 150, np.float64(1025871.1260015869)), (123, 149, np.float64(1000943.1167047119))]
